# 松尾研 Physical AI｜実写17枚・極キャップ合成＋入口ゲート校正版 V2（配布修正版）
作成日時：2026/10/29 16:04 JST

前回の極キャップ合成版はケース02で停止した。V2は入口ゲートの開発用校正のみを更新した版である。ZIP内の原本17枚は1セットのみ保存し、実行開始時にSHA-256検証して作業用コピーを再生成する。元の検査コードと物理制御コードは変更しない。

**注意：** ケース01・02の実カメラ画像を校正に含むため、そのスコアは独立検証ではない。等級判定・物理制御は変更していない。10/10成功は未確認である。

Google Drive接続は不要。次のZIPを選択する：`PAI_REAL17_POLAR_GATECAL_V2_COMPACT_R1_20261029.zip`

In [ ]:
from pathlib import Path
import subprocess
font_candidates=[Path('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'),Path('/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc')]
if not any(p.is_file() for p in font_candidates):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','fonts-noto-cjk'],check=True)
print('CJK_FONT_READY',next((str(p) for p in font_candidates if p.is_file()),'NOT_FOUND'),flush=True)


In [ ]:
from pathlib import Path
import sys, shutil, zipfile, subprocess, json, hashlib
from google.colab import files
prefix='PAI_REAL17_POLAR_GATECAL_V2_COMPACT_R1_20261029'
exact=Path('/content/'+prefix+'.zip')
matches=sorted(Path('/content').glob(prefix+'*.zip'))
if exact.is_file(): name=exact
elif matches: name=matches[-1]
else:
    print('ZIP選択：'+prefix+'.zip',flush=True)
    uploaded=files.upload()
    assert len(uploaded)==1,'ZIPを1つだけ選択してください'
    uploaded_name=next(iter(uploaded))
    assert uploaded_name.startswith(prefix) and uploaded_name.endswith('.zip'),'V2 COMPACT R1 ZIPを選択してください'
    name=Path('/content')/uploaded_name
assert name.is_file() and name.name.startswith(prefix) and name.suffix=='.zip','V2 COMPACT R1 ZIPを指定してください'
root=Path('/content/PAI_REAL17_POLAR_GATECAL_V2_STAGE88')
if root.exists(): shutil.rmtree(root)
root.mkdir(parents=True)
with zipfile.ZipFile(name) as z:
    assert z.testzip() is None,'ZIP CRC破損'
    for n in z.namelist():
        p=Path(n)
        assert not p.is_absolute() and '..' not in p.parts,'安全でないZIPパス'
    z.extractall(root)
# Recreate redundant working copies from the canonical 17-photo ZIP; never modify source assets.
raw=root/'REAL17_USER_SOURCE_17.zip'
manifest=json.loads((root/'realphoto/asset_swap_manifest.json').read_text(encoding='utf-8'))
expected=manifest['original_photo_sha256']
with zipfile.ZipFile(raw) as z:
    assert z.testzip() is None and len(z.namelist())==17,'SOURCE_17_ZIP_INVALID'
    assert set(z.namelist())==set(expected),'SOURCE_17_MANIFEST_NAME_MISMATCH'
    source_dir=root/'REAL17_SOURCE_ORIGINAL_17'
    runtime_dir=root/'realphoto/original_images'
    source_dir.mkdir(parents=True,exist_ok=True); runtime_dir.mkdir(parents=True,exist_ok=True)
    for n in z.namelist():
        p=Path(n); assert not p.is_absolute() and '..' not in p.parts and len(p.parts)==1,'UNSAFE_SOURCE_PATH'
        data=z.read(n); digest=hashlib.sha256(data).hexdigest()
        assert digest==expected[n], 'SOURCE_SHA256_MISMATCH '+n
        (source_dir/n).write_bytes(data); (runtime_dir/n).write_bytes(data)
print('CANONICAL_17_SOURCE_HASH_VERIFIED_AND_WORKING_COPIES_CREATED',len(expected),flush=True)
subprocess.run([sys.executable,str(root/'scripts/check_direct_source.py')],cwd=root,check=True)
print('REAL17_POLAR_GATECAL_V2_COMPACT_R1_READY',root,'ZIP_BYTES',name.stat().st_size,flush=True)


In [ ]:
cmd=[sys.executable,'-u',str(root/'run_realphoto_full10.py'),'--repo',str(root),'--out','/content/PAI_REAL17_POLAR_GATECAL_V2_RESULTS','--install']
rc=subprocess.run(cmd,cwd=root,check=False).returncode
print('REAL17_POLAR_GATECAL_V2_PHYSICS_RETURN',rc,'0=10/10合格, 2=実験未達, 3=環境・検査エラー',flush=True)
result=Path('/content/PAI_G8_REAL17_FULL10_RESULT.zip')
if result.is_file():
    print('RESULT_READY',result,result.stat().st_size,flush=True)
    files.download(str(result))
else: print('RESULT_ZIP_NOT_CREATED; 上の実行ログを保存してください',flush=True)
